# Adaptive Learning — Matematika SD
**BKT + Ontologi | Cindy Anatasya Sagala | IPB University**

Jalankan cell **satu per satu dari atas ke bawah**.

---
## 1. Install & Upload File

In [ ]:
!pip install flask networkx -q
print('✅ Dependencies installed')

In [ ]:
# Upload semua file project
from google.colab import files

print('Upload file berikut (pilih semua sekaligus):')
print('  ontology.py, bkt_engine.py, data_generator.py,')
print('  param_estimator.py, evaluator.py,')
print('  database.py, seed_questions.py, app.py')
print('  data/math_grade1.json')
print()

uploaded = files.upload()
print(f'\n✅ {len(uploaded)} file diupload: {list(uploaded.keys())}')

In [ ]:
import os

# Buat folder data/ dan pindahkan math_grade1.json ke sana
os.makedirs('data', exist_ok=True)
if os.path.exists('math_grade1.json'):
    os.rename('math_grade1.json', 'data/math_grade1.json')

# Verifikasi semua file ada
required = [
    'ontology.py', 'bkt_engine.py', 'data_generator.py',
    'param_estimator.py', 'evaluator.py',
    'database.py', 'seed_questions.py', 'app.py',
    'data/math_grade1.json'
]
missing = [f for f in required if not os.path.exists(f)]
if missing:
    print('❌ File tidak ditemukan:', missing)
else:
    print('✅ Semua file siap')

### Upload index.html (frontend)

In [ ]:
# Upload index.html (file frontend)
import os
os.makedirs('static', exist_ok=True)

print('Upload 1 file: index.html')
uploaded2 = files.upload()

if 'index.html' in uploaded2:
    # Taruh di dua tempat supaya pasti ketemu
    import shutil
    shutil.copy('index.html', 'static/index.html')
    print('✅ index.html siap')
else:
    print('❌ Upload index.html gagal')


---
## 2. Generate Data Sintetis

In [ ]:
%run data_generator.py

---
## 3. Estimasi Parameter BKT

In [ ]:
%run param_estimator.py

---
## 4. Evaluasi Model (BKT+Ontologi vs Baseline)

In [ ]:
%run evaluator.py

---
## 5. Inisialisasi Database & Seed Soal

In [ ]:
from database import init_db
init_db()
print('✅ Database initialized')

%run seed_questions.py

---
## 6. Jalankan Platform (Flask + Tunnel)
Cell ini akan print **URL publik**. Buka URL itu di browser untuk mengakses platform.

In [ ]:
import threading, time, subprocess, re

# Download cloudflared
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

# Jalankan Flask di background
def run_flask():
    import importlib, app as app_module
    importlib.reload(app_module)
    app_module.app.run(port=5000, use_reloader=False, quiet=True)

flask_thread = threading.Thread(target=run_flask, daemon=True)
flask_thread.start()
time.sleep(3)

# Test Flask dulu sebelum tunnel
import urllib.request
try:
    urllib.request.urlopen('http://localhost:5000')
    print('✅ Flask berjalan di port 5000')
except Exception as e:
    print('❌ Flask belum siap:', e)

# Buka tunnel
tunnel = subprocess.Popen(
    ['./cloudflared', 'tunnel', '--url', 'http://localhost:5000'],
    stderr=subprocess.PIPE,
    stdout=subprocess.DEVNULL
)

print('Menunggu tunnel (maks 30 detik)...')
url_found = None
for line in tunnel.stderr:
    line = line.decode('utf-8', errors='ignore').strip()
    # cloudflared menulis URL di stderr dalam format berbeda tergantung versi
    match = re.search(r'https://[\w\-]+\.trycloudflare\.com', line)
    if match:
        url_found = match.group()
        break

if url_found:
    print('\n' + '='*55)
    print('🌐 Buka link ini di browser (klik atau copy-paste):')
    print(f'   {url_found}')
    print('='*55)
    print('Jangan tutup cell ini selama demo berlangsung.')
else:
    print('❌ URL tunnel tidak ditemukan.')
    print('Coba jalankan ulang cell ini.')


---
## (Opsional) Cek isi database

In [ ]:
from database import count_questions, get_conn

print(f'Total soal di DB: {count_questions()}')

with get_conn() as conn:
    rows = conn.execute(
        'SELECT kc_id, COUNT(*) as n FROM questions GROUP BY kc_id ORDER BY kc_id'
    ).fetchall()

for row in rows:
    print(f'  {row["kc_id"]}: {row["n"]} soal')

In [ ]:
# Cek siswa yang sudah terdaftar
from database import get_conn

with get_conn() as conn:
    rows = conn.execute(
        'SELECT id, name, total_stars, created_at FROM students ORDER BY created_at DESC LIMIT 10'
    ).fetchall()

if rows:
    print(f'{'ID':<10} {'Nama':<15} {'Bintang':>8}')
    print('-' * 35)
    for r in rows:
        print(f'{r["id"]:<10} {r["name"]:<15} {r["total_stars"]:>8}')
else:
    print('Belum ada siswa terdaftar.')

In [ ]:
# Stop tunnel jika sudah selesai demo
tunnel.terminate()
print('Tunnel ditutup.')


---
## 7. Monitor Siswa (jalankan saat testing berlangsung)

In [ ]:
# Jalankan cell ini untuk lihat progress siswa secara live
# Bisa dijalankan berulang selama sesi testing
from database import get_conn

with get_conn() as conn:
    students = conn.execute("""
        SELECT s.id, s.name, s.total_stars,
               COUNT(i.id) as n_interaksi,
               ROUND(AVG(i.correct)*100, 1) as akurasi_pct,
               COUNT(DISTINCT CASE WHEN ks.is_mastered=1 THEN ks.kc_id END) as kc_mastered
        FROM students s
        LEFT JOIN interactions i ON i.student_id = s.id
        LEFT JOIN kc_states ks ON ks.student_id = s.id
        GROUP BY s.id
        ORDER BY s.created_at
    """).fetchall()

print(f"{'Nama':<15} {'Interaksi':>10} {'Akurasi':>8} {'KC Selesai':>11} {'Bintang':>8}")
print('-' * 58)
for s in students:
    acc = f"{s['akurasi_pct']}%" if s['akurasi_pct'] else '-'
    print(f"{s['name']:<15} {s['n_interaksi']:>10} {acc:>8} {s['kc_mastered']:>11} {s['total_stars']:>8}")
print(f"\nTotal siswa: {len(students)}")


---
## 8. Export Data Setelah Testing Selesai

In [ ]:
%run export_data.py

# Download CSV ke komputer
from google.colab import files
import os

for f in ['data/real_interactions.csv', 'data/real_students.csv',
          'data/real_kc_states.csv', 'data/adaptive_learning.db']:
    if os.path.exists(f):
        files.download(f)
        print(f'⬇️  Downloading {f}')


---
## 9. Re-train Model dari Data Nyata (jalankan setelah testing)

In [ ]:
# Re-estimasi parameter BKT dari data interaksi nyata
# Jalankan SETELAH export_data.py
%run retrain.py

# Download parameter baru
from google.colab import files
files.download('data/estimated_params.json')
print('\n⬇️  estimated_params.json didownload.')
print('Simpan file ini dan upload ke Colab saat demo berikutnya.')
